# Statistical Calibration and Conditional Power

Recompute the historical inference examples, 28 planning scenarios, 390,000 Student/Welch attempts, 100,000 known-bound attempts and 120,000 empirical-Bernstein attempts for the capped finite task. See the [method settings and output map](README.md). These methods and their negative calibration findings are distinct from budget-only nominal inference. These are synthetic statistical computations, not human-study evidence or theoretical prediction validation.

Requires a checkout, Python dependencies installed below, and Node.js for deterministic schedule generation. The supplementary fixed-500 paired Gamma diagnostic below uses Python only. All inputs are generated from the scientific settings.

For the paired diagnostic reference, create and select the Python 3.10.9 environment in `configs/native_reference_environment.yml` before opening this notebook. Set `PRESERVE_PAIRED_REFERENCE = True` below to retain NumPy 1.23.5 / SciPy 1.10.0 during installation. This is the diagnostic runtime, not the earlier calibration runtime (3.12.14 / 2.3.5 / 1.16.3); other platforms are not promised bitwise equivalence.

In [ ]:
from pathlib import Path
import subprocess
import shutil
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'src').is_dir())
PRESERVE_PAIRED_REFERENCE = False  # Set True in the paired diagnostic reference kernel.
install = [sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)]
if PRESERVE_PAIRED_REFERENCE:
    install += ['-c', str(ROOT / 'configs/native_reference_constraints.txt')]
subprocess.check_call(install)
sys.path.insert(0, str(ROOT / 'src'))
if shutil.which('node') is not None:
    subprocess.run(['node', '--version'], check=True)
else:
    print('Install Node.js for the schedule panels; the paired Gamma diagnostic does not need it.')

## Settings and Scope

`smoke` uses the same seeds and 250-attempt batch zero of every cell; it is wiring only and overlaps full. `full` preserves every frozen cell, stage, policy, and criterion. Large computations are disabled. Outputs are written under ignored `results/`; use a fresh destination for each run.

The historical result retains MIXED/111 failures. The bounded and variance-adaptive sensitivities neither replace the primary method nor recover original-population targets under informative selection. Raw floating truths are not snapped to an ideal zero.

In [ ]:
import json
import shlex
RUN = False
MODE = 'smoke'  # 'full' preserves 390000 + 100000 + 120000 attempts
OUTPUT = Path('results/statistical_reproduction') / MODE
assert MODE in ('smoke', 'full')
configs = {
    'student_welch': 'configs/tutoring_calibration_v01.json',
    'bounded': 'configs/tutoring_bounded_robustness_v01.json',
    'variance_adaptive': 'configs/tutoring_identifiability_robustness_v01.json',
    'planning': 'configs/tutoring_power_sensitivity_v01.json',
}
for name, path in configs.items():
    spec = json.loads((ROOT / path).read_text())
    print(name, path)
    print(json.dumps(spec, indent=2))
print('Node.js:', shutil.which('node'))

In [ ]:
commands = []
for analysis in ('inference', 'planning', 'bounded', 'variance-adaptive'):
    commands.append([sys.executable, 'scripts/reproduce_tutoring_statistics.py', '--analysis', analysis, '--mode', MODE, '--output', str(OUTPUT / analysis)])
commands.append([sys.executable, 'scripts/calibrate_tutoring_methods.py', '--config', configs['student_welch'], '--mode', MODE, '--output', str(OUTPUT / 'student_welch')])
for command in commands:
    print(shlex.join(command))
if RUN:
    if shutil.which('node') is None:
        raise RuntimeError('Install Node.js before deterministic schedule preparation')
    for command in commands:
        subprocess.run(command, cwd=ROOT, check=True)

## Read Newly Computed Results

The reader below only opens outputs from this notebook's producer commands. It never accepts an external prepared context or historical result folder. Inspect all CSV/JSON diagnostics rather than treating successful process exit as all criteria passing. The bounded and variance-adaptive summaries retain method support, eligibility, paired comparisons, null rosters, selection diagnostics and failed criteria.

In [ ]:
if RUN:
    calibration = json.loads((ROOT / OUTPUT / 'student_welch/calibration_summary.json').read_text())
    planning = json.loads((ROOT / OUTPUT / 'planning/planning.json').read_text())
    print('Conditional planning scenarios:', len(planning['scenarios']))
    print('Inference examples:', [p.name for p in sorted((ROOT / OUTPUT / 'inference').glob('*.json'))])
    failures = json.loads((ROOT / OUTPUT / 'student_welch/criteria_readback.json').read_text())
    print('Student/Welch:', calibration['attempts'], 'attempts;', len(failures['failureRecords']), 'failed criteria;', failures['status'])
    for analysis in ('bounded', 'variance-adaptive'):
        result = json.loads((ROOT / OUTPUT / analysis / 'summary.json').read_text())
        print(analysis, result['attempts'], result['status'], 'failures:', len(result['criteriaFailures']))
    context = json.loads((ROOT / OUTPUT / 'variance-adaptive/prepared_context.json').read_text())
    print('Recomputed projection corners:', context['cornerCoefficients'])
    for name in ('C1_states.json', 'cohort_identification.json', 'source_slope_domain.json'):
        artifact = json.loads((ROOT / OUTPUT / 'variance-adaptive' / name).read_text())
        print(name, 'recomputed artifact entries:', len(artifact))
    for cell in context['cells']:
        truth = cell['truths']['selectedCore']
        print(cell['id'], 'raw nulls:', truth['rawNullRoster'], 'ideal nulls:', truth['idealNullRoster'])

## Fixed-500 Paired Gamma Diagnostic

This supplementary same-law/same-input attribution compares frozen legacy Student inference with current nominal H05 on the identical original person slopes, then separately compares canonical reprojection. The public producer retains gamma_skew_null, seed 2026100627, complete C1-C5 draw order, batches 0 and 39 of 250 cases each, 20 C4 people and 10 complete trials. It does not clip the abstract unbounded Gamma responses into native allocations.

The independent reader regenerates all raw inputs, derives centered OLS support/projections, and uses its own Student arithmetic with scipy.special. It checks four Student outputs per case, all identities, raw arrays and aggregate counts. It does not call the producer's estimators. The generator is shared only as the input-law primitive.

RUN_PAIRED is independent of RUN above. Keep both false for a dry command preview. Select [0] or [39] for 250-case verification without changing RNG shape or fixed identities. Selected results explicitly remain incomplete for fixed500. Outputs require new directories; nothing is overwritten.

In [ ]:
RUN_PAIRED = False
PAIRED_BATCHES = [0, 39]  # [0] or [39] selects one original 250-case batch.
PAIRED_OUTPUT = Path('results/paired_gamma_diagnostic')
paired_runner = 'scripts/reproduce_paired_diagnostic.py'
paired_producer = PAIRED_OUTPUT / 'producer'
paired_reader = PAIRED_OUTPUT / 'reader'
assert PAIRED_BATCHES in ([0], [39], [0, 39])
paired_commands = [
    [sys.executable, paired_runner, 'produce', '--batches', *map(str, PAIRED_BATCHES), '--output', str(paired_producer)],
    [sys.executable, paired_runner, 'read', '--input', str(paired_producer), '--output', str(paired_reader)],
]
if PAIRED_BATCHES == [0, 39]:
    paired_commands[1].append('--require-full')
for command in paired_commands:
    print(shlex.join(command))
if RUN_PAIRED:
    from tutoring_analysis.paired_recipe import runtime, REFERENCE_RUNTIME
    if runtime() != REFERENCE_RUNTIME:
        raise RuntimeError('Select the Python 3.10.9 reference kernel and preserve its pinned NumPy/SciPy versions in the first cell.')
    for command in paired_commands:
        subprocess.run(command, cwd=ROOT, check=True)
    paired_result = json.loads((ROOT / paired_reader / 'summary.json').read_text())
    print({k: paired_result[k] for k in ('cases', 'selectedComplete', 'fixed500Complete', 'aggregateDifferences', 'actualMaxErrors')})

### Interpretation and Existing Packets

The original reference diagnostic observed 40/500 rejections and 471/500 intervals covering zero in each arm, with zero paired rejection or coverage disagreements and nonzero roundoff differences. These are reference observations, not cached notebook outputs or pass thresholds. Legacy rejection uses p <= .05; current and projected rejection use p < .05. Boundary counts remain explicit.

The reader can also inspect an existing original producer packet via `read --input PACKET --output NEW_DIRECTORY --require-full`. It checks the scientific arrays/cases/plan/tests/summary without modifying input records. Numerical readback does not authenticate the original runtime.

Agreement does not establish Type-I calibration, reproduce original-runtime bits, add 500 independent historical attempts, repair the historical 111 failures, or validate native-task/human inference or cap20 transfer.

## Limits

No universal Type-I guarantee, human behavior, final recruitment N, original-population MNAR repair, or new theory is inferred. Historical global-Holm and supporting alternatives retain their original policies. This notebook has no UI replay/normalization dependency and does not claim to reproduce prior report graphics.